<a href="https://colab.research.google.com/github/momo25bend/streamsentinel/blob/main/streamsentinel_J2_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Installation, GPU, Drive

In [26]:
!pip install -q transformers

import torch, os, glob, json
import cv2
import numpy as np
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil utilisé :", device)

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/streamsentinel"
DOSSIERS = {k: f"{BASE}/{k}" for k in ["photos", "masques", "resultats", "anonymisees"]}
for d in DOSSIERS.values():
    os.makedirs(d, exist_ok=True)

EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
photos = sorted(p for ext in EXTENSIONS for p in glob.glob(f"{DOSSIERS['photos']}/{ext}"))
print(len(photos), "photos trouvées")

TAILLE_MAX = 1024

def charger(chemin):
    img = ImageOps.exif_transpose(Image.open(chemin)).convert("RGB")
    img.thumbnail((TAILLE_MAX, TAILLE_MAX))
    return img

Appareil utilisé : cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
51 photos trouvées


1

In [27]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

def _vecteur(sortie, projection):
    # selon la version de transformers, on reçoit un tenseur ou un objet
    v = sortie if isinstance(sortie, torch.Tensor) else sortie.pooler_output
    # on projette seulement si la dimension ne correspond pas encore
    if v.shape[-1] != projection.out_features:
        v = projection(v)
    return v / v.norm(dim=-1, keepdim=True)

@torch.no_grad()
def vecteur_image(img_pil):
    entrees = clip_processor(images=img_pil, return_tensors="pt").to(device)
    return _vecteur(clip_model.get_image_features(**entrees), clip_model.visual_projection)

@torch.no_grad()
def vecteurs_texte(phrases):
    entrees = clip_processor(text=phrases, return_tensors="pt",
                             padding=True, truncation=True).to(device)
    return _vecteur(clip_model.get_text_features(**entrees), clip_model.text_projection)

# vérification : les deux doivent afficher 512
img_test = charger(photos[0])
print("image :", vecteur_image(img_test).shape[-1])
print("texte :", vecteurs_texte(["a photo of a river"]).shape[-1])

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

image : 512
texte : 512


## 2. Validation : est-ce bien un cours d'eau ?

On compare l'image à une liste de scènes possibles et on regarde laquelle
correspond le mieux. Les scènes « pièges » (piscine, mer, rue, intérieur)
sont indispensables : sans elles, tout finirait classé comme rivière.

In [28]:
SCENES = {
    "cours_deau":   ["a photo of a river", "a photo of a small stream in nature",
                     "a photo of an urban canal", "a photo of a creek with rocks"],
    "plan_deau":    ["a photo of a lake", "a photo of the sea and a beach",
                     "a photo of a swimming pool"],
    "hors_sujet":   ["a photo of a street with cars", "a photo of a parking lot",
                     "a photo of a building", "an indoor photo of a room",
                     "a portrait photo of a person", "a photo of a forest without water",
                     "a screenshot of a document or a map"],
}

PHRASES, GROUPES = [], []
for groupe, phrases in SCENES.items():
    PHRASES += phrases
    GROUPES += [groupe] * len(phrases)

VECTEURS_SCENES = vecteurs_texte(PHRASES)

def valider_scene(img_pil, seuil=0.35):
    similarites = (vecteur_image(img_pil) @ VECTEURS_SCENES.T)[0]
    probas = torch.softmax(similarites * 100, dim=0).cpu().numpy()   # 100 = échelle interne de CLIP

    # probabilité cumulée par groupe
    par_groupe = {}
    for g, p in zip(GROUPES, probas):
        par_groupe[g] = par_groupe.get(g, 0.0) + float(p)

    meilleur = PHRASES[int(np.argmax(probas))]
    score = par_groupe["cours_deau"]
    return {
        "est_cours_deau": bool(score >= seuil),
        "score_cours_deau": round(score, 3),
        "scores": {g: round(v, 3) for g, v in par_groupe.items()},
        "scene_reconnue": meilleur,
    }

for p in photos[:3]:
    print(os.path.basename(p), valider_scene(charger(p)))

berge_betonnee_00.jpg {'est_cours_deau': True, 'score_cours_deau': 0.974, 'scores': {'cours_deau': 0.974, 'plan_deau': 0.017, 'hors_sujet': 0.009}, 'scene_reconnue': 'a photo of an urban canal'}
berge_betonnee_01.jpg {'est_cours_deau': True, 'score_cours_deau': 0.496, 'scores': {'cours_deau': 0.496, 'plan_deau': 0.344, 'hors_sujet': 0.16}, 'scene_reconnue': 'a photo of a swimming pool'}
berge_betonnee_02.jpg {'est_cours_deau': False, 'score_cours_deau': 0.0, 'scores': {'cours_deau': 0.0, 'plan_deau': 0.0, 'hors_sujet': 1.0}, 'scene_reconnue': 'a photo of a parking lot'}


### Test sur toutes les photos de rivières et les photos hors sujet

In [29]:
import pandas as pd

lignes = []
for p in photos:
    nom = os.path.basename(p)
    if nom.startswith("dechets"):
        continue
    try:
        r = valider_scene(charger(p))
    except Exception as e:
        print("Ignorée :", nom, type(e).__name__)
        continue
    lignes.append({
        "photo": nom,
        "attendu": "non" if nom.startswith(("sans_eau", "rate_")) else "oui",
        "detecte": "oui" if r["est_cours_deau"] else "non",
        "score": r["score_cours_deau"],
        "scene": r["scene_reconnue"],
    })

validation = pd.DataFrame(lignes)
validation["correct"] = validation["attendu"] == validation["detecte"]
print("Précision :", round(validation["correct"].mean() * 100, 1), "%")
display(validation)

Précision : 81.0 %


,photo,attendu,detecte,score,scene,correct
0,berge_betonnee_00.jpg,oui,oui,0.974,a photo of an urban canal,True
1,berge_betonnee_01.jpg,oui,oui,0.496,a photo of a swimming pool,True
2,berge_betonnee_02.jpg,oui,non,0.000,a photo of a parking lot,False
3,berge_naturelle_00.jpg,oui,oui,0.558,a photo of a forest without water,True
4,berge_naturelle_01.jpg,oui,oui,0.989,a photo of a small stream in nature,True
5,berge_naturelle_02.jpg,oui,oui,0.575,a photo of a small stream in nature,True
6,eau_boueuse_00.jpg,oui,oui,0.475,a photo of a river,True
7,eau_boueuse_01.jpg,oui,oui,0.907,a photo of a river,True
8,eau_boueuse_02.jpg,oui,oui,0.994,a photo of a river,True
9,eau_verte_00.jpg,oui,non,0.181,a screenshot of a document or a map,False


## 3. Vérifier la description de l'utilisateur

L'utilisateur écrit en français, CLIP comprend l'anglais. On traduit donc
les **mots-clés** du français vers des phrases de référence anglaises, puis on
demande à CLIP de trancher entre « l'élément est là » et « il n'y est pas ».

Cette approche est plus fiable que de comparer directement la phrase entière,
et elle indique précisément **quel élément** ne correspond pas.

In [42]:
# mot-clé français -> (phrase si présent, phrase si absent)
ELEMENTS = {
    "mousse":    ("white foam floating on the water surface", "a water surface without foam"),
    "déchet":    ("plastic trash floating in the water", "a clean river without trash"),
    "bouteille": ("a plastic bottle in the water", "a river without any bottle"),
    "poisson":   ("a dead fish floating on the water", "a river without any fish"),
    "algue":     ("green algae covering the water", "clear water without algae"),
    "boueuse":   ("brown muddy turbid water", "clear transparent water"),
    "huile":     ("an oily rainbow film on the water", "a water surface without oil"),
    "béton":     ("a concrete channel bank", "a natural vegetated river bank"),
    "tuyau":     ("a drain pipe discharging into the river", "a river bank without any pipe"),
    "mort":      ("a dead fish floating on the water", "a river without any fish"),
}

# variantes d'écriture -> mot-clé
SYNONYMES = {
    "mousse": "mousse", "mousseux": "mousse", "écume": "mousse",
    "déchet": "déchet", "déchets": "déchet", "ordure": "déchet", "plastique": "déchet",
    "bouteille": "bouteille", "bouteilles": "bouteille", "canette": "déchet",
    "poisson": "poisson", "poissons": "poisson", "mort": "mort", "morts": "mort",
    "algue": "algue", "algues": "algue", "verte": "algue", "vert": "algue",
    "boueuse": "boueuse", "boueux": "boueuse", "trouble": "boueuse", "marron": "boueuse",
    "huile": "huile", "hydrocarbure": "huile", "irisation": "huile", "irisé": "huile",
    "béton": "béton", "bétonnée": "béton", "bétonné": "béton", "mur": "béton",
    "tuyau": "tuyau", "buse": "tuyau", "rejet": "tuyau", "canalisation": "tuyau",
}

def elements_declares(description):
    mots = "".join(c.lower() if c.isalpha() or c.isspace() else " " for c in description).split()
    trouves = {SYNONYMES[m] for m in mots if m in SYNONYMES}
    return sorted(trouves)

def verifier_description(img_pil, description, seuil=0.25):
    declares = elements_declares(description)
    if not declares:
        return {"verifiable": False,
                "message": "Description trop vague pour être vérifiée automatiquement."}

    vec_img = vecteur_image(img_pil)
    details, incoherents = {}, []
    for cle in declares:
        present, absent = ELEMENTS[cle]
        sims = (vec_img @ vecteurs_texte([present, absent]).T)[0]
        proba = float(torch.softmax(sims * 100, dim=0)[0])
        details[cle] = round(proba, 3)
        if proba < seuil:
            incoherents.append(cle)

    return {
        "verifiable": True,
        "elements_declares": declares,
        "confiance_par_element": details,
        "coherente": len(incoherents) == 0,
        "non_confirmes": incoherents,
    }

# exemples
exemple = charger([p for p in photos if "mousse_01" in p][0])
print(verifier_description(exemple, "il y a de la mousse blanche sur l'eau"))
print(verifier_description(exemple, "beaucoup de bouteilles en plastique flottent"))

{'verifiable': True, 'elements_declares': ['mousse'], 'confiance_par_element': {'mousse': 0.324}, 'coherente': True, 'non_confirmes': []}
{'verifiable': True, 'elements_declares': ['bouteille', 'déchet'], 'confiance_par_element': {'bouteille': 0.184, 'déchet': 0.099}, 'coherente': False, 'non_confirmes': ['bouteille', 'déchet']}


## 4. Floutage des visages et des plaques (RGPD)

Une photo prise en ville peut contenir des passants ou des véhicules. Le
règlement européen impose de ne pas diffuser ces données personnelles.

On utilise les détecteurs de Haar fournis avec OpenCV : rapides, sans
téléchargement, mais limités aux visages **de face** et aux plaques bien
visibles. Pour la production, un modèle dédié serait nécessaire — à écrire
dans les limites du README.

In [31]:
FLOUTAGE_DISPO = hasattr(cv2, "CascadeClassifier") and hasattr(cv2, "data")

if FLOUTAGE_DISPO:
    detecteur_visages = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    detecteur_plaques = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_russian_plate_number.xml")
    FLOUTAGE_DISPO = not (detecteur_visages.empty() or detecteur_plaques.empty())

print("Floutage disponible :", FLOUTAGE_DISPO)

def flouter_zone(image, x, y, w, h):
    zone = image[y:y + h, x:x + w]
    if zone.size == 0:
        return
    k = max(11, (min(w, h) // 2) * 2 + 1)
    image[y:y + h, x:x + w] = cv2.GaussianBlur(zone, (k, k), 0)

def anonymiser(img_pil):
    if not FLOUTAGE_DISPO:
        return img_pil, {"visages": 0, "plaques": 0, "indisponible": True}

    image = np.array(img_pil).copy()
    gris = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    visages = detecteur_visages.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=6, minSize=(30, 30))
    plaques = detecteur_plaques.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=6, minSize=(40, 15))

    for (x, y, w, h) in list(visages) + list(plaques):
        flouter_zone(image, x, y, w, h)

    return Image.fromarray(image), {"visages": len(visages), "plaques": len(plaques)}

total = {"visages": 0, "plaques": 0}
for p in photos:
    try:
        img, compte = anonymiser(charger(p))
    except Exception:
        continue
    if compte.get("visages") or compte.get("plaques"):
        img.save(f"{DOSSIERS['anonymisees']}/{os.path.basename(p)}")
        print(os.path.basename(p), compte)
    total["visages"] += compte.get("visages", 0)
    total["plaques"] += compte.get("plaques", 0)

print("Total :", total)

Floutage disponible : False
Total : {'visages': 0, 'plaques': 0}


In [24]:
import cv2
print("version :", cv2.__version__)
print("fichier :", cv2.__file__)
print("cascadeClassifier présent :", hasattr(cv2, "CascadeClassifier"))
print("cv2.data présent :", hasattr(cv2, "data"))

!pip uninstall -y opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-contrib-python-headless


version : 5.0.0
fichier : /usr/local/lib/python3.13/dist-packages/cv2/__init__.py
cascadeClassifier présent : False
cv2.data présent : True


In [25]:
!pip install -q "opencv-contrib-python-headless<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 MB 13.0 MB/s eta 0:00:00


⚠️ **À vérifier à l'œil** : ouvrez le dossier `anonymisees` sur votre Drive.
Les détecteurs de Haar produisent des fausses détections (un rocher pris pour
un visage). Ce n'est pas grave ici — flouter un rocher ne gêne personne — mais
un visage **manqué** est un vrai problème. Notez-le dans les limites.

## 5. Messages de reprise

Quand une photo est rejetée, l'app doit dire **quoi corriger**, pas seulement
« photo invalide ». Chaque cause produit une consigne concrète.

In [32]:
SEUIL_FLOU, SEUIL_SOMBRE, SEUIL_CLAIR = 100, 50, 210

def controle_qualite(img_pil):
    gris = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2GRAY)
    nettete = float(cv2.Laplacian(gris, cv2.CV_64F).var())
    luminosite = float(gris.mean())
    problemes = []
    if nettete < SEUIL_FLOU:
        problemes.append(("flou", "Photo floue : calez vos coudes et refaites la photo."))
    if luminosite < SEUIL_SOMBRE:
        problemes.append(("sombre", "Photo trop sombre : cherchez un endroit plus éclairé."))
    if luminosite > SEUIL_CLAIR:
        problemes.append(("surexpose", "Photo éblouie : tournez le dos au soleil."))
    return {"nettete": round(nettete, 1), "luminosite": round(luminosite, 1),
            "valide": not problemes, "problemes": problemes}

def messages_reprise(qualite, scene, description=None):
    consignes = [m for _, m in qualite["problemes"]]

    if not scene["est_cours_deau"]:
        if scene["scores"].get("plan_deau", 0) > 0.3:
            consignes.append("Ce plan d'eau ne semble pas être un cours d'eau. "
                             "StreamSentinel ne couvre que rivières, ruisseaux et canaux.")
        else:
            consignes.append("Aucun cours d'eau détecté. Cadrez la rivière en remplissant "
                             "au moins un tiers de l'image.")
    elif scene["score_cours_deau"] < 0.6:
        consignes.append("Le cours d'eau est peu visible : rapprochez-vous de la berge.")

    if description and description.get("verifiable") and not description["coherente"]:
        manquants = ", ".join(description["non_confirmes"])
        consignes.append(f"Nous ne retrouvons pas sur la photo : {manquants}. "
                         "Prenez une seconde photo plus proche de cet élément.")

    return consignes or ["Photo acceptée, analyse en cours."]

## 6. Chaîne de validation complète

In [43]:
def valider_photo(chemin, description=""):
    img = charger(chemin)
    resultat = {"photo": os.path.basename(chemin)}

    qualite = controle_qualite(img)
    resultat["qualite"] = qualite
    if not qualite["valide"]:
        resultat["acceptee"] = False
        resultat["consignes"] = [m for _, m in qualite["problemes"]]
        return resultat

    img_anonyme, compte = anonymiser(img)
    resultat["anonymisation"] = compte

    scene = valider_scene(img_anonyme)
    resultat["scene"] = scene

    verif = verifier_description(img_anonyme, description) if description else None
    if verif:
        resultat["description"] = verif

    resultat["acceptee"] = bool(scene["est_cours_deau"])
    resultat["consignes"] = messages_reprise(qualite, scene, verif)
    return resultat

# démonstration sur 4 cas
for nom, texte in [("mousse_01.jpg", "de la mousse blanche flotte sur l'eau"),
                   ("mousse_01.jpg", "un poisson mort au bord"),
                   ("sans_eau_00.jpg", "la rivière est sale"),
                   ("rate_flou.jpg", "eau trouble")]:
    chemin = f"{DOSSIERS['photos']}/{nom}"
    if os.path.exists(chemin):
        r = valider_photo(chemin, texte)
        print(f"\n--- {nom} | « {texte} »")
        print("Acceptée :", r["acceptee"])
        for c in r["consignes"]:
            print("  →", c)


--- mousse_01.jpg | « de la mousse blanche flotte sur l'eau »
Acceptée : True
  → Photo acceptée, analyse en cours.

--- mousse_01.jpg | « un poisson mort au bord »
Acceptée : True
  → Nous ne retrouvons pas sur la photo : mort, poisson. Prenez une seconde photo plus proche de cet élément.

--- sans_eau_00.jpg | « la rivière est sale »
Acceptée : False
  → Aucun cours d'eau détecté. Cadrez la rivière en remplissant au moins un tiers de l'image.

--- rate_flou.jpg | « eau trouble »
Acceptée : False
  → Photo floue : calez vos coudes et refaites la photo.


In [40]:
img = charger(f"{DOSSIERS['photos']}/mousse_01.jpg")
for texte in ["de la mousse blanche flotte sur l'eau", "un poisson mort au bord",
              "beaucoup de bouteilles en plastique"]:
    print(texte, "→", verifier_description(img, texte)["confiance_par_element"])

de la mousse blanche flotte sur l'eau → {'mousse': 0.324}
un poisson mort au bord → {'mort': 0.006, 'poisson': 0.006}
beaucoup de bouteilles en plastique → {'bouteille': 0.184, 'déchet': 0.099}


## 7. Enregistrement des résultats du J2

In [44]:
from tqdm.auto import tqdm

resultats_j2, ignorees = [], []
for p in tqdm(photos):
    if os.path.basename(p).startswith("dechets"):
        continue
    try:
        resultats_j2.append(valider_photo(p))
    except Exception as e:
        ignorees.append((os.path.basename(p), type(e).__name__))

with open(f"{DOSSIERS['resultats']}/resultats_j2.json", "w", encoding="utf-8") as f:
    json.dump(resultats_j2, f, ensure_ascii=False, indent=2)

acceptees = sum(r["acceptee"] for r in resultats_j2)
print(f"{len(resultats_j2)} photos traitées, {acceptees} acceptées, {len(ignorees)} ignorées")

  0%|          | 0/51 [00:00<?, ?it/s]

21 photos traitées, 13 acceptées, 0 ignorées


## 8. En parallèle : entraîner le détecteur de déchets

Cette cellule tourne seule pendant 30 à 60 minutes. Lancez-la pendant que vous
travaillez sur le reste — mais dans un **second notebook Colab**, car un seul
notebook n'exécute qu'une cellule à la fois.

Le modèle final est enregistré sur le Drive : il survivra à la fin de session.

In [ ]:
!pip install -q ultralytics roboflow

from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dechets = (rf.workspace("rf100-vl").project("floating-waste-8deje-lrbq")
             .version(2).download("yolov8", location="/content/datasets/dechets"))

modele = YOLO("yolo11s.pt")
modele.train(
    data=f"{dechets.location}/data.yaml",
    epochs=40, imgsz=640, batch=16, patience=10,
    project=f"{BASE}/modeles", name="dechets",
)